**Part 1 — Python & Data Foundations: ingest, validate, clean, enrich**

Task 1.1

In [8]:
import csv
import json

In [9]:
with open("appointments.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    print("Headers:", reader.fieldnames)

    rows = list(reader)

    print("Total rows:", len(rows))

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [10]:
def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

In [11]:
accepted = []
rejected = []

for row in rows:
    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

In [12]:
with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

In [13]:
# 5. Print results
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("Rejected rows saved to rejected_rows.json")

Accepted rows: 988
Rejected rows: 32
Rejected rows saved to rejected_rows.json


Task 1.2

In [14]:
import pandas as pd

In [15]:
df = pd.read_csv("appointments.csv")
df.head(5)

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status
0,A0001,P509,C05,2025-01-24,2025-01-25,60.0,F,18.0,850,Phone,Consultation,Yes,1,No-Show
1,A0002,P046,C07,2025-05-27,2025-06-06,39.0,M,12.5,350,Walk-in,Follow-up,No,2,Attended
2,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
3,A0003,P009,C02,2025-03-24,2025-03-30,63.0,F,4.1,850,Walk-in,Consultation,No,0,Attended
4,A0004,P482,C01,2025-04-18,2025-04-22,52.0,F,7.5,750,App,Consultation,Yes,0,Attended


In [16]:
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(1020, 14)


In [17]:
print("\nDataset Information:")
df.info()


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [18]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


In [19]:
print("\nDuplicate Rows:")
print(df.duplicated().sum())


Duplicate Rows:
20


In [20]:
# Drop exact duplicate rows
df = df.drop_duplicates()
print("\nRows after dropping duplicates:", len(df))


Rows after dropping duplicates: 1000


In [21]:
# Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after cleaning booking_channel:", len(df))

Rows after cleaning booking_channel: 1000


In [22]:
# Convert date columns
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)
print("Rows after date conversion:", len(df))
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)
print("Rows after date conversion:", len(df))

Rows after date conversion: 1000
Rows after date conversion: 1000


In [23]:
df = df.dropna(subset=["appt_date", "status"])
print("Rows after dropping missing dates/status:", len(df))

Rows after dropping missing dates/status: 990


In [24]:
# Final dataset inspection
print("\nFinal Dataset Shape:", df.shape)
print("\nFinal Missing Values:")
print(df.isnull().sum())


Final Dataset Shape: (990, 14)

Final Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           0
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              0
dtype: int64


Task 1.3

In [25]:
import numpy as np

In [26]:
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Invalid ages replaced with missing values.")

Invalid ages replaced with missing values.


In [27]:
df["age"] = df["age"].fillna(
    df["age"].median()
).round().astype(int)

In [28]:
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

In [29]:
print("\nMissing values filled.")
print("Missing age:", df["age"].isnull().sum())
print("Missing distance:", df["distance_km"].isnull().sum())


Missing values filled.
Missing age: 0
Missing distance: 0


In [30]:
# Distance_km - Outlier treatment
Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

# Count values outside the fences before capping
distance_capped = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
).sum()

# Cap outliers
df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

In [31]:
# Fee_inr - - Outlier treatment
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

# Count values outside the fences before capping
fee_capped = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
).sum()

# Cap outliers
df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

In [32]:
# Print capped values
print("\nOutlier Treatment Results:")
print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)


Outlier Treatment Results:
Distance values capped: 26
Fee values capped: 3


In [33]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

In [34]:
print("\nFinal Data Types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

print("\nFinal Dataset:")
print(df[["age", "distance_km", "fee_inr"]].head())


Final Data Types:
age              int64
distance_km    float64
fee_inr          int64
dtype: object

Final Dataset:
   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700


Task 1.4

In [35]:
# Convert both columns to datetime
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], errors="coerce")


print(df[["appt_date", "booked_on"]].dtypes)

appt_date    datetime64[ns]
booked_on    datetime64[ns]
dtype: object


In [36]:
df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

In [37]:
df["no_show"] = np.where(
    df["status"] == "No-Show", 1, 0
)

In [38]:
# Results
print(df[["appt_date", "booked_on", "lead_days", "no_show"]].head())

   appt_date  booked_on  lead_days  no_show
0 2025-01-25 2025-01-24          1        1
1 2025-06-06 2025-05-27         10        0
2 2025-03-30 2025-03-24          6        0
4 2025-04-22 2025-04-18          4        0
5 2025-01-02 2024-11-23         40        1


Task 1.5

In [39]:
import requests

In [40]:
clinics = pd.read_csv("clinics.csv")
clinics_enriched = clinics.copy()

In [41]:
def get_temperature(latitude, longitude, clinic_name):

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    reason = "Unknown error"

    # Try maximum 2 times (1 attempt + 1 retry)
    for attempt in range(2):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            # Check status
            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            # Read JSON response
            data = response.json()

            # Check if hourly data exists
            if "hourly" not in data:
                reason = "Hourly data missing"
                continue

            temperatures = data["hourly"].get("temperature_2m")

            if not temperatures:
                reason = "Temperature data missing or empty"
                continue

            # Take the FIRST hourly temperature (midnight UTC)
            temp_c = temperatures[0]

            return temp_c

        except requests.exceptions.RequestException as e:
            reason = str(e)

        except ValueError as e:
            reason = f"Invalid JSON response: {e}"

    # Both attempts failed
    print(f"Failed for {clinic_name}: {reason}")

    return None

In [42]:
# Fetch temperature for all 8 clinics
temp_values = []

for index, row in clinics_enriched.iterrows():

    temperature = get_temperature(
        row["latitude"],
        row["longitude"],
        row["clinic_name"]
    )

    temp_values.append(temperature)


clinics_enriched["temp_c"] = temp_values



clinics_enriched.to_csv(
    "clinics_enriched.csv",
    index=False
)


print("\nEnriched Clinics Dataset:")
print(clinics_enriched)

print("\nSaved successfully as clinics_enriched.csv")
print("Total clinics:", len(clinics_enriched))
print("Missing temperatures:", clinics_enriched["temp_c"].isnull().sum())


Enriched Clinics Dataset:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    19.4  
1    19.7  
2    19.7  
3    16.4  
4    16.4  
5    26.3  
6    16.1  
7    14.7  

Saved successfully as clinics_enriched.csv
Total clinics: 8
Missing temperatures: 0


Task 1.6

In [43]:
import numbers

In [44]:
appointment_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

In [45]:
appointments_export = df[appointment_columns].copy()

In [46]:
appointments_export = appointments_export.sort_values(
    by="appointment_id"
).reset_index(drop=True)

In [47]:
appointments_export["booked_on"] = (
    appointments_export["booked_on"].dt.strftime("%Y-%m-%d")
)

appointments_export["appt_date"] = (
    appointments_export["appt_date"].dt.strftime("%Y-%m-%d")
)

In [48]:
is_unique = appointments_export["appointment_id"].is_unique

print("Appointment IDs are unique:", is_unique)

# Check missing values in all 16 columns
missing_values = appointments_export.isnull().sum()

print("\nMissing values per column:")
print(missing_values)

no_missing = missing_values.sum() == 0

print("\nNo missing values:", no_missing)

Appointment IDs are unique: True

Missing values per column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

No missing values: True


In [49]:
appointments_export.to_csv(
    "appointments_clean.csv",
    index=False
)

print("\napppointments_clean.csv saved successfully!")
print("Final shape:", appointments_export.shape)


apppointments_clean.csv saved successfully!
Final shape: (990, 16)


In [50]:
clinics_enriched = pd.read_csv("clinics_enriched.csv")

print("\nClinics dataset:")
print(clinics_enriched.head())


Clinics dataset:
  clinic_id             clinic_name    city    tier  latitude  longitude  \
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077   
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260   
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882   
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150   
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873   

   temp_c  
0    19.4  
1    19.7  
2    19.7  
3    16.4  
4    16.4  


In [51]:
# SQL VALUE CONVERSION FUNCTION
def sql_value(value):

    if pd.isna(value):
        return "NULL"

    if isinstance(value, (bool, np.bool_)):
        return "1" if value else "0"

    if isinstance(value, numbers.Number):
        return str(value)

    return "'" + str(value).replace("'", "''") + "'"

In [52]:
with open("load_data.sql", "w", encoding="utf-8") as file:

    # --------------------------------------
    # FIRST: INSERT CLINICS
    # --------------------------------------

    print("\nWriting clinic INSERT statements...")

    for _, row in clinics_enriched.iterrows():

        values = [
            sql_value(value)
            for value in row
        ]

        sql = f"INSERT INTO clinics VALUES ({','.join(values)});\n"

        file.write(sql)

    # --------------------------------------
    # SECOND: INSERT APPOINTMENTS
    # --------------------------------------

    print("Writing appointment INSERT statements...")

    for _, row in appointments_export.iterrows():

        values = [
            sql_value(value)
            for value in row
        ]

        sql = f"INSERT INTO appointments VALUES ({','.join(values)});\n"

        file.write(sql)


Writing clinic INSERT statements...
Writing appointment INSERT statements...


In [53]:
print("\nAll tasks completed successfully!")

print("Appointments exported:", len(appointments_export))
print("Clinics exported:", len(clinics_enriched))

print("\nFiles created:")
print("1. appointments_clean.csv")
print("2. load_data.sql")


All tasks completed successfully!
Appointments exported: 990
Clinics exported: 8

Files created:
1. appointments_clean.csv
2. load_data.sql
